# 01. 온통청년 정책 데이터 전처리 기준 정리 및 실행

이 노트북은 `youth_policies_categorized.csv`와 `youth_policies_summary_by_region_category.csv`를 불러와서  
분석용 정책 데이터셋을 만드는 단계입니다.

## 전처리 기준

| 구분 | 처리 기준 |
|---|---|
| 입력 데이터 | 온통청년 API로 수집한 정책 단위 CSV |
| 분석 단위 | 지역 × 정책 |
| 중복 제거 | `지역 + 정책ID` 기준 우선 제거, 정책ID가 없는 경우 `지역 + 정책명` 기준 제거 |
| 결측 처리 | 텍스트 컬럼은 빈 문자열로 대체, 수치/날짜 파생 컬럼은 별도 계산 |
| 분석 텍스트 | 정책명, 정책키워드, 정책설명, 정책지원내용, 지원대상, 신청방법, 주관기관, 운영기관을 결합 |
| 분류 기준 | `대표분류` 우선 사용, 없으면 `자동분류` 사용 |
| URL 접근성 | 신청URL, 참고URL1, 참고URL2 중 하나라도 존재하면 URL 있음 |
| 날짜 처리 | 신청기간/사업기간에서 `YYYYMMDD` 형식 날짜를 추출하여 시작일/종료일 파생 |
| 산출물 | 전처리 완료 CSV, 지역×분류 피벗표, 전처리 요약표 |

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 80)

DATA_DIR = Path(".")
OUTPUT_DIR = DATA_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

POLICY_FILE = DATA_DIR / "youth_policies_categorized.csv"
SUMMARY_FILE = DATA_DIR / "youth_policies_summary_by_region_category.csv"

print("현재 작업 폴더:", DATA_DIR.resolve())
print("정책 파일 존재:", POLICY_FILE.exists())
print("요약 파일 존재:", SUMMARY_FILE.exists())

현재 작업 폴더: C:\Users\yong\Desktop\TM\tm
정책 파일 존재: True
요약 파일 존재: True


In [2]:
def read_csv_auto(path: Path) -> pd.DataFrame:
    """CSV 인코딩이 달라도 최대한 안정적으로 읽는 함수."""
    encodings = ["utf-8-sig", "utf-8", "cp949", "euc-kr"]
    last_error = None

    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception as e:
            last_error = e

    raise last_error

if not POLICY_FILE.exists():
    raise FileNotFoundError(
        "youth_policies_categorized.csv 파일을 현재 노트북과 같은 폴더에 넣어 주세요."
    )

df_raw = read_csv_auto(POLICY_FILE)
summary_raw = read_csv_auto(SUMMARY_FILE) if SUMMARY_FILE.exists() else None

print("원본 정책 데이터 크기:", df_raw.shape)
display(df_raw.head())

원본 정책 데이터 크기: (5470, 27)


,지역,조회_zipCd,정책ID,정책명,정책키워드,정책설명,정책지원내용,정책대분류,정책중분류,자동분류,대표분류,신청기간,사업기간,지원대상,나이조건,소득조건,신청방법,제출서류,주관기관,운영기관,신청URL,참고URL1,참고URL2,최초등록일시,최종수정일시,수집페이지,수집출처
0,서울,11000,20260605005400113228,청년미래적금,보조금,"청년들의 기초자산 형성을 지원하기 위한 정책형 금융상품으로, 3년 만기 시까지 매월 최대 50만원 한도 내에서 자유롭게 납입 가능(202...","은행이자+비과세 혜택+정부기여금(납입금액에 비례해 일반형 6%, 우대형 12%의 정부기여금 지원)",금융･복지･문화,취약계층 및 금융지원,복지,복지,20260622 ~ 20261231,20260622 ~ 20261231,"19세~34세 / 연령제한:N 0043003 0 0 총급여 7,500만원 이하 또는 연매출 3억원 이하 소상공인 중 가구 중위소득 200...",19세~34세 / 연령제한:N,"0043003 0 0 총급여 7,500만원 이하 또는 연매출 3억원 이하 소상공인 중 가구 중위소득 200% 이하인 청년을 대상",ㅇ 취급은행 모바일앱을 통해 매월 비대면 신청 가능 ㅇ 2026년 6월 출시 예정,NaN,금융위원회,한국고용정보원,NaN,https://www.kinfa.or.kr/financialProduct/youthFutureSavings.do,https://blog.naver.com/blogfsc/224302863400,2026-06-05 18:06:49,2026-06-10 14:05:31,1,온통청년_OPEN_API_getPlcy
1,서울,11000,20260528005400113227,(농식품부) 농식품 바우처,바우처,『농업·농촌 및 식품산업 기본법』 제 23조의 2(취약계층 등에 대한 식품지원)에 근거하여 취약계층의 식품 접근성을 강화하고 균형 있는 ...,"- 지원방식: 전자바우처(카드방식) - 지원품목: 국산 채소류, 과일류, 육류, 신선알류, 흰 우유, 잡곡류, 두부류, 임산물 (이외 품...",금융･복지･문화,건강,복지,복지,20251222 ~ 20261211,20260102 ~ 20261231,18세~34세 / 연령제한:N 0043003 0 0 생계급여(기준 중위소득 32%이하) 수급가구 중 임산부·영유아·아동·청년 포함가구 생...,18세~34세 / 연령제한:N,0043003 0 0 생계급여(기준 중위소득 32%이하) 수급가구 중 임산부·영유아·아동·청년 포함가구,1. 방문신청 : 주소지 관할 읍ㆍ면ㆍ동 행정복지센터 방문 2. 전화신청 : 고객지원센터 (1551-0857)를 통해 신청 3. 온라인신...,NaN,농림축산식품부,한국고용정보원,https://www.foodvoucher.go.kr/security/joinAgree,https://www.foodvoucher.go.kr/view/fm/vucintro/agriFood,NaN,2026-05-28 10:00:50,2026-05-28 10:01:12,1,온통청년_OPEN_API_getPlcy
2,서울,11000,20260528005400113226,(문체부) 청년예술인 예술활동 적립계좌,보조금,청년예술인에게 중장기 자산형성의 기회를 마련하여 안정적인 예술활동을 지원하는 사업,매월 일정 금액을 24개월간 적금 저축 시 가입자가 저축한 금액만큼 정부지원금을 지원 (1인 최대 240만원) - 상품종류: 10만원 정...,금융･복지･문화,예술인지원,복지,복지,NaN,20260101 ~ 20261231,18세~39세 / 연령제한:N 0043002 0 3692 「예술인복지법」상 예술활동증명을 완료한 예술인(신청일 기준 예술활동증명 유효자)...,18세~39세 / 연령제한:N,0043002 0 3692,1. 예술활동증명확인 예술인경력정보시스템(https://www.kawfartist.kr)을 접속하시어 경력지원 > 예술활동증명 > 신청내...,1. 주민등록초본 - 2026년 발급분 - 발급 시 발급대상자 본인 및 전체 발급 2. 소득금액증명원 - 2026년 발급분 - 귀속년도:...,문화체육관광부,한국고용정보원,https://www.artloan.kr/notice/savingsAccountProcess.do,https://www.artloan.kr/notice/savingsAccount.do,NaN,2026-05-28 09:34:16,2026-05-28 09:34:49,1,온통청년_OPEN_API_getPlcy
3,서울,11000,20260527005400113224,청년 국가기술자격 응시료 지원 사업,보조금,"구직활동을 하거나 경력을 개발하는 청년들의 경제적 부담을 완화하고, 국가기술자격 취득을 통한 취업 경쟁력 강화를 도모",34세 이하 청년*이(소득 및 취업 여부 무관) 한국산업인력공단이 시행하는 국가기술자격 시험(’26년 기준 491종목)에 응시하는 경우 ...,일자리,취업,일자리,일자리,NaN,상시,0세~34세 / 연령제한:N 0043001 0 0 0011009 0013010 0049010 0055003,0세~34세 / 연령제한:N,0043001 0 0,"한국산업인력공단 Q-Net(https://www.q-net.or.kr) 원서접수 결제 단계에서, 34세 이하 청년인 경우 별도의 신청 절...",NaN,고용노동부,한국산업인력공단,https://www.q-net.or.kr/man001.do?gSite=Q&gIntro=Y,https://www.q-net.or.kr/man004.do?id=man00402&gSite=Q&gId=,NaN,2026-05-27 11:16:43,2026-05-27 11:17:06,1,온통청년_OPEN_API_getPlcy
4,서울,11000,20260527005400113223,전세보증금반환보증 보증료 지원,주거지원,"전세사기, 역전세 등 임차인이 전세보증금을 돌려받지 못하는 전세 피해를 예방하고, 상대적으로 주거 취약계층인 청년 및 저소득층의 주거 안...","○ 지원대상 - 신청일 기준 유효한 전세보증금반환보증(HUG, HF, SGI)에 가입한 임차보증금 3억원 이하, 연소득 (청년) 5천만원...",주거,전월세 및 주거급여 지원,"주거지원, 복지",주거지원,NaN,상시,0세~0세 / 연령제한:Y 0043001 0 0 0011009 0013010 0049010 0055003,0세~0세 / 연령제한:Y,0043001 0 0,○ 신청방법 : 시·군·구청 또는 주민센터에 방문 접수 혹은 온라인 접수 ※ 방문접수의 경우 구군별 상이하니 전화 후 방문 필요 * 대구...,"○ 보증기관에서 보증가입 시 전세보증금반환보증 보증료 지원 사업을 위한 제3자 정보제공에 동의한 경우 - 보증료 지원 신청서, 서약서 ○...",국토교통부,국토교통부,https://www.gov.kr/portal/rcvfvrSvc/dtlEx/161300000103,https://www.gov.kr/portal/rcvfvrSvc/dtlEx/161300000103,NaN,2026-05-27 11:06:12,2026-05-27 11:06:36,1,온통청년_OPEN_API_getPlcy


In [3]:
# 필수 컬럼이 없더라도 노트북이 멈추지 않게 안전하게 컬럼을 보정합니다.
required_cols = [
    "지역", "조회_zipCd", "정책ID", "정책명", "정책키워드", "정책설명", "정책지원내용",
    "정책대분류", "정책중분류", "자동분류", "대표분류", "신청기간", "사업기간",
    "지원대상", "나이조건", "소득조건", "신청방법", "제출서류",
    "주관기관", "운영기관", "신청URL", "참고URL1", "참고URL2",
    "최초등록일시", "최종수정일시", "수집페이지", "수집출처"
]

df = df_raw.copy()

for col in required_cols:
    if col not in df.columns:
        df[col] = np.nan

# 문자열 컬럼 정리
text_like_cols = [
    "지역", "조회_zipCd", "정책ID", "정책명", "정책키워드", "정책설명", "정책지원내용",
    "정책대분류", "정책중분류", "자동분류", "대표분류", "신청기간", "사업기간",
    "지원대상", "나이조건", "소득조건", "신청방법", "제출서류",
    "주관기관", "운영기관", "신청URL", "참고URL1", "참고URL2",
    "최초등록일시", "최종수정일시", "수집출처"
]

def clean_basic_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = x.replace("\u200b", " ").replace("\xa0", " ")
    x = re.sub(r"\s+", " ", x)
    return x.strip()

for col in text_like_cols:
    df[col] = df[col].map(clean_basic_text)

print("컬럼 보정 후 크기:", df.shape)
display(df.head(3))

컬럼 보정 후 크기: (5470, 27)


,지역,조회_zipCd,정책ID,정책명,정책키워드,정책설명,정책지원내용,정책대분류,정책중분류,자동분류,대표분류,신청기간,사업기간,지원대상,나이조건,소득조건,신청방법,제출서류,주관기관,운영기관,신청URL,참고URL1,참고URL2,최초등록일시,최종수정일시,수집페이지,수집출처
0,서울,11000,20260605005400113228,청년미래적금,보조금,"청년들의 기초자산 형성을 지원하기 위한 정책형 금융상품으로, 3년 만기 시까지 매월 최대 50만원 한도 내에서 자유롭게 납입 가능(202...","은행이자+비과세 혜택+정부기여금(납입금액에 비례해 일반형 6%, 우대형 12%의 정부기여금 지원)",금융･복지･문화,취약계층 및 금융지원,복지,복지,20260622 ~ 20261231,20260622 ~ 20261231,"19세~34세 / 연령제한:N 0043003 0 0 총급여 7,500만원 이하 또는 연매출 3억원 이하 소상공인 중 가구 중위소득 200...",19세~34세 / 연령제한:N,"0043003 0 0 총급여 7,500만원 이하 또는 연매출 3억원 이하 소상공인 중 가구 중위소득 200% 이하인 청년을 대상",ㅇ 취급은행 모바일앱을 통해 매월 비대면 신청 가능 ㅇ 2026년 6월 출시 예정,,금융위원회,한국고용정보원,,https://www.kinfa.or.kr/financialProduct/youthFutureSavings.do,https://blog.naver.com/blogfsc/224302863400,2026-06-05 18:06:49,2026-06-10 14:05:31,1,온통청년_OPEN_API_getPlcy
1,서울,11000,20260528005400113227,(농식품부) 농식품 바우처,바우처,『농업·농촌 및 식품산업 기본법』 제 23조의 2(취약계층 등에 대한 식품지원)에 근거하여 취약계층의 식품 접근성을 강화하고 균형 있는 ...,"- 지원방식: 전자바우처(카드방식) - 지원품목: 국산 채소류, 과일류, 육류, 신선알류, 흰 우유, 잡곡류, 두부류, 임산물 (이외 품...",금융･복지･문화,건강,복지,복지,20251222 ~ 20261211,20260102 ~ 20261231,18세~34세 / 연령제한:N 0043003 0 0 생계급여(기준 중위소득 32%이하) 수급가구 중 임산부·영유아·아동·청년 포함가구 생...,18세~34세 / 연령제한:N,0043003 0 0 생계급여(기준 중위소득 32%이하) 수급가구 중 임산부·영유아·아동·청년 포함가구,1. 방문신청 : 주소지 관할 읍ㆍ면ㆍ동 행정복지센터 방문 2. 전화신청 : 고객지원센터 (1551-0857)를 통해 신청 3. 온라인신...,,농림축산식품부,한국고용정보원,https://www.foodvoucher.go.kr/security/joinAgree,https://www.foodvoucher.go.kr/view/fm/vucintro/agriFood,,2026-05-28 10:00:50,2026-05-28 10:01:12,1,온통청년_OPEN_API_getPlcy
2,서울,11000,20260528005400113226,(문체부) 청년예술인 예술활동 적립계좌,보조금,청년예술인에게 중장기 자산형성의 기회를 마련하여 안정적인 예술활동을 지원하는 사업,매월 일정 금액을 24개월간 적금 저축 시 가입자가 저축한 금액만큼 정부지원금을 지원 (1인 최대 240만원) - 상품종류: 10만원 정...,금융･복지･문화,예술인지원,복지,복지,,20260101 ~ 20261231,18세~39세 / 연령제한:N 0043002 0 3692 「예술인복지법」상 예술활동증명을 완료한 예술인(신청일 기준 예술활동증명 유효자)...,18세~39세 / 연령제한:N,0043002 0 3692,1. 예술활동증명확인 예술인경력정보시스템(https://www.kawfartist.kr)을 접속하시어 경력지원 > 예술활동증명 > 신청내...,1. 주민등록초본 - 2026년 발급분 - 발급 시 발급대상자 본인 및 전체 발급 2. 소득금액증명원 - 2026년 발급분 - 귀속년도:...,문화체육관광부,한국고용정보원,https://www.artloan.kr/notice/savingsAccountProcess.do,https://www.artloan.kr/notice/savingsAccount.do,,2026-05-28 09:34:16,2026-05-28 09:34:49,1,온통청년_OPEN_API_getPlcy


In [4]:
# 중복 제거
before = len(df)

df["정책ID_정리"] = df["정책ID"].replace({"nan": "", "None": ""}).fillna("").astype(str).str.strip()
df["정책명_정리"] = df["정책명"].fillna("").astype(str).str.strip()

# 최종수정일시가 있으면 최신 자료를 우선 남깁니다.
df["_최종수정일시_dt"] = pd.to_datetime(df["최종수정일시"], errors="coerce")
df = df.sort_values(["지역", "_최종수정일시_dt"], ascending=[True, False])

has_id = df["정책ID_정리"].ne("")
df_with_id = df[has_id].drop_duplicates(subset=["지역", "정책ID_정리"], keep="first")
df_without_id = df[~has_id].drop_duplicates(subset=["지역", "정책명_정리"], keep="first")

df = pd.concat([df_with_id, df_without_id], ignore_index=True)
after = len(df)

print(f"중복 제거 전: {before:,}건")
print(f"중복 제거 후: {after:,}건")
print(f"제거된 중복: {before - after:,}건")

중복 제거 전: 5,470건
중복 제거 후: 5,470건
제거된 중복: 0건


In [5]:
# 대표분류/자동분류 정리
CATEGORY_ORDER = ["일자리", "직무교육", "주거지원", "창업지원", "복지", "참여 프로그램", "기타"]

def normalize_category(row):
    base = row.get("대표분류", "")
    if not base:
        base = row.get("자동분류", "")
    base = clean_basic_text(base)

    # 자동분류가 "일자리, 직무교육"처럼 복수일 경우 첫 번째 분류를 대표로 사용
    if "," in base:
        base = base.split(",")[0].strip()

    aliases = {
        "주거": "주거지원",
        "주택": "주거지원",
        "교육": "직무교육",
        "취업교육": "직무교육",
        "창업": "창업지원",
        "참여": "참여 프로그램",
        "프로그램": "참여 프로그램",
        "복지지원": "복지",
        "금융": "복지",
        "문화": "복지",
    }

    if base in CATEGORY_ORDER:
        return base

    for k, v in aliases.items():
        if k in base:
            return v

    return "기타"

df["대표분류_정리"] = df.apply(normalize_category, axis=1)

display(df["대표분류_정리"].value_counts().rename_axis("대표분류_정리").reset_index(name="정책수"))

,대표분류_정리,정책수
0,일자리,3155
1,직무교육,1305
2,복지,585
3,주거지원,324
4,참여 프로그램,83
5,창업지원,18


In [6]:
# 분석용 텍스트 결합
analysis_text_cols = [
    "정책명", "정책키워드", "정책설명", "정책지원내용",
    "지원대상", "나이조건", "소득조건", "신청방법",
    "주관기관", "운영기관"
]

def remove_code_noise(text):
    """정책 API에서 섞여 들어오는 7자리 분류 코드 등 분석 잡음을 일부 제거합니다."""
    text = clean_basic_text(text)
    text = re.sub(r"\b\d{7}\b", " ", text)  # 예: 0011009 같은 코드
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["분석텍스트_원본"] = df[analysis_text_cols].fillna("").agg(" ".join, axis=1).map(clean_basic_text)
df["분석텍스트"] = df["분석텍스트_원본"].map(remove_code_noise)
df["분석텍스트길이"] = df["분석텍스트"].str.len()

# URL 존재 여부
url_cols = ["신청URL", "참고URL1", "참고URL2"]
df["URL존재여부"] = df[url_cols].apply(lambda row: any(str(x).strip() for x in row), axis=1).astype(int)

# 지원대상/신청방법/제출서류 등 정보 구체성 보조 지표
df["지원대상존재여부"] = df["지원대상"].str.len().gt(0).astype(int)
df["신청방법존재여부"] = df["신청방법"].str.len().gt(0).astype(int)
df["제출서류존재여부"] = df["제출서류"].str.len().gt(0).astype(int)

display(df[["지역", "정책명", "대표분류_정리", "분석텍스트길이", "URL존재여부"]].head())

,지역,정책명,대표분류_정리,분석텍스트길이,URL존재여부
0,강원,청년미래적금,복지,380,1
1,강원,스마트 모빌리티 창업캠프사업,일자리,430,1
2,강원,산림산업 창업지원_청년 임팩트 창업 아이디어 챌린지,일자리,520,1
3,강원,(농식품부) 농식품 바우처,복지,795,1
4,강원,삼척형 청년인턴 지원사업,일자리,341,1


In [7]:
# 신청기간/사업기간 날짜 파생
def extract_period_dates(period_text):
    text = clean_basic_text(period_text)
    dates = re.findall(r"(20\d{6})", text)

    if not dates:
        return pd.NaT, pd.NaT

    start = pd.to_datetime(dates[0], format="%Y%m%d", errors="coerce")
    end = pd.to_datetime(dates[-1], format="%Y%m%d", errors="coerce")
    return start, end

df[["신청시작일", "신청종료일"]] = df["신청기간"].apply(lambda x: pd.Series(extract_period_dates(x)))
df[["사업시작일", "사업종료일"]] = df["사업기간"].apply(lambda x: pd.Series(extract_period_dates(x)))

today = pd.Timestamp.today().normalize()
df["신청기간_날짜존재"] = df["신청시작일"].notna().astype(int)
df["현재신청가능추정"] = (
    df["신청시작일"].notna()
    & df["신청종료일"].notna()
    & (df["신청시작일"] <= today)
    & (today <= df["신청종료일"])
).astype(int)

display(df[["정책명", "신청기간", "신청시작일", "신청종료일", "현재신청가능추정"]].head())

,정책명,신청기간,신청시작일,신청종료일,현재신청가능추정
0,청년미래적금,20260622 ~ 20261231,2026-06-22,2026-12-31,0
1,스마트 모빌리티 창업캠프사업,20260401 ~ 20260528,2026-04-01,2026-05-28,0
2,산림산업 창업지원_청년 임팩트 창업 아이디어 챌린지,20260520 ~ 20260603,2026-05-20,2026-06-03,0
3,(농식품부) 농식품 바우처,20251222 ~ 20261211,2025-12-22,2026-12-11,1
4,삼척형 청년인턴 지원사업,20260202 ~ 20260206,2026-02-02,2026-02-06,0


In [8]:
# 지역별/분류별 요약표
region_summary = (
    df.groupby("지역", as_index=False)
      .agg(
          정책수=("정책명", "count"),
          평균텍스트길이=("분석텍스트길이", "mean"),
          URL존재율=("URL존재여부", "mean"),
          신청기간기재율=("신청기간_날짜존재", "mean"),
          현재신청가능추정비율=("현재신청가능추정", "mean"),
      )
      .sort_values("정책수", ascending=False)
)

category_summary = (
    df.groupby("대표분류_정리", as_index=False)
      .agg(
          정책수=("정책명", "count"),
          평균텍스트길이=("분석텍스트길이", "mean"),
          URL존재율=("URL존재여부", "mean"),
      )
      .sort_values("정책수", ascending=False)
)

region_category_pivot = (
    pd.pivot_table(
        df,
        index="지역",
        columns="대표분류_정리",
        values="정책명",
        aggfunc="count",
        fill_value=0
    )
    .reindex(columns=CATEGORY_ORDER, fill_value=0)
)

display(region_summary)
display(category_summary)
display(region_category_pivot)

,지역,정책수,평균텍스트길이,URL존재율,신청기간기재율,현재신청가능추정비율
8,충남,707,350.902405,0.816124,0.437058,0.084866
7,제주,608,430.748355,0.800987,0.495066,0.049342
2,경남,584,421.356164,0.833904,0.414384,0.017123
3,경북,539,442.753247,0.788497,0.504638,0.031540
5,전남,535,421.442991,0.753271,0.454206,0.024299
6,전북,534,430.966292,0.720974,0.464419,0.039326
1,경기,515,420.939806,0.811650,0.473786,0.054369
9,충북,498,437.212851,0.751004,0.437751,0.020080
0,강원,489,435.200409,0.754601,0.488753,0.030675
4,서울,461,478.548807,0.750542,0.462039,0.019523


,대표분류_정리,정책수,평균텍스트길이,URL존재율
1,일자리,3155,447.120444,0.810777
3,직무교육,1305,342.902682,0.754789
0,복지,585,452.948718,0.747009
2,주거지원,324,475.969136,0.669753
4,참여 프로그램,83,402.746988,0.686747
5,창업지원,18,415.388889,0.944444


대표분류_정리,일자리,직무교육,주거지원,창업지원,복지,참여 프로그램,기타
지역,,,,,,,
강원,297,114,21,1,48,8,0
경기,300,124,28,1,54,8,0
경남,341,128,40,2,67,6,0
경북,326,117,38,1,48,9,0
서울,274,112,22,1,46,6,0
전남,300,132,30,4,62,7,0
전북,309,123,29,2,62,9,0
제주,345,149,40,3,63,8,0
충남,368,189,49,2,83,16,0


In [9]:
# 저장
preprocessed_path = OUTPUT_DIR / "policy_preprocessed.csv"
region_summary_path = OUTPUT_DIR / "policy_region_summary.csv"
category_summary_path = OUTPUT_DIR / "policy_category_summary.csv"
pivot_path = OUTPUT_DIR / "policy_region_category_pivot.csv"

df.to_csv(preprocessed_path, index=False, encoding="utf-8-sig")
region_summary.to_csv(region_summary_path, index=False, encoding="utf-8-sig")
category_summary.to_csv(category_summary_path, index=False, encoding="utf-8-sig")
region_category_pivot.to_csv(pivot_path, encoding="utf-8-sig")

print("저장 완료")
print("-", preprocessed_path)
print("-", region_summary_path)
print("-", category_summary_path)
print("-", pivot_path)

저장 완료
- outputs\policy_preprocessed.csv
- outputs\policy_region_summary.csv
- outputs\policy_category_summary.csv
- outputs\policy_region_category_pivot.csv


## 다음 단계

전처리 결과가 정상적으로 저장되었다면 다음 노트북인  
`02_policy_scoring_mismatch.ipynb`를 실행하면 됩니다.